In [1]:
import sys
import platform

print("Python:", sys.version)
print("Architecture:", platform.machine())
print("Interpreter:", sys.executable)

Python: 3.12.16 (main, Oct 25 2026, 18:24:37) [Clang 21.0.0 (clang-2100.3.34.2)]
Architecture: arm64
Interpreter: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/bin/python


In [2]:
import sys
import platform

print("Python version:", sys.version)
print("Architecture:", platform.machine())
print("Interpreter:", sys.executable)

Python version: 3.12.16 (main, Oct 25 2026, 18:24:37) [Clang 21.0.0 (clang-2100.3.34.2)]
Architecture: arm64
Interpreter: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/bin/python


In [3]:
import importlib.util

packages = [
    "numpy",
    "pandas",
    "rasterio",
    "h5py",
    "matplotlib",
    "geopandas",
    "ipykernel"
]

for package in packages:
    status = "INSTALLED" if importlib.util.find_spec(package) else "MISSING"
    print(f"{package:15} {status}")

numpy           INSTALLED
pandas          INSTALLED
rasterio        INSTALLED
h5py            INSTALLED
matplotlib      INSTALLED
geopandas       MISSING
ipykernel       INSTALLED


In [4]:
from pathlib import Path

# Find the repository root
ROOT = Path.cwd()

while not (
    (ROOT / "L_BAND").exists()
    and (ROOT / "S_BAND").exists()
):
    if ROOT.parent == ROOT:
        raise FileNotFoundError(
            "Could not locate L_BAND and S_BAND folders."
        )
    ROOT = ROOT.parent

print("Repository root:", ROOT)
print("\nL-band folder:", ROOT / "L_BAND")
print("S-band folder:", ROOT / "S_BAND")
print("L+S folder:", ROOT / "L_PLUS_S")

Repository root: /Users/shrenishetty/Documents/NISAR_L+S/NISAR

L-band folder: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/L_BAND
S-band folder: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/S_BAND
L+S folder: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/L_PLUS_S


In [5]:
import pandas as pd

extensions = {".tif", ".tiff", ".h5", ".hdf5", ".json", ".csv"}

files = []

for band in ["L_BAND", "S_BAND"]:
    band_folder = ROOT / band

    for path in band_folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            files.append({
                "Band": band,
                "Filename": path.name,
                "Extension": path.suffix.lower(),
                "Relative path": str(path.relative_to(ROOT)),
                "Size (MB)": round(path.stat().st_size / (1024**2), 2)
            })

inventory = pd.DataFrame(files)

display(inventory.sort_values(["Band", "Extension", "Filename"]))

print("\nTotal files:", len(inventory))
print("\nFiles by band:")
print(inventory["Band"].value_counts())
print("\nFiles by extension:")
print(inventory["Extension"].value_counts())

,Band,Filename,Extension,Relative path,Size (MB)
6,L_BAND,classification_thresholds.csv,.csv,L_BAND/L BAND/metadata/classification_threshol...,0.00
9,L_BAND,hdf5_dataset_inventory.csv,.csv,L_BAND/L BAND/metadata/hdf5_dataset_inventory.csv,0.35
10,L_BAND,validation_metrics.csv,.csv,L_BAND/L BAND/validation/validation_metrics.csv,0.00
8,L_BAND,water_area_statistics.csv,.csv,L_BAND/L BAND/metadata/water_area_statistics.csv,0.00
7,L_BAND,run_summary.json,.json,L_BAND/L BAND/metadata/run_summary.json,0.00
14,L_BAND,thresholds_and_scene_report.json,.json,L_BAND/L BAND/classification/20260819/threshol...,0.00
19,L_BAND,thresholds_and_scene_report.json,.json,L_BAND/L BAND/classification/20260924/threshol...,0.00
24,L_BAND,thresholds_and_scene_report.json,.json,L_BAND/L BAND/classification/20261018/threshol...,0.00
2,L_BAND,18_HHHH_gamma0_db.tif,.tif,L_BAND/L BAND/extracted features/18_HHHH_gamma...,14.31
3,L_BAND,18_HVHV_gamma0_db.tif,.tif,L_BAND/L BAND/extracted features/18_HVHV_gamma...,13.81



Total files: 41

Files by band:
Band
L_BAND    26
S_BAND    15
Name: count, dtype: int64

Files by extension:
Extension
.tif     27
.csv      8
.json     6
Name: count, dtype: int64


In [6]:
import rasterio
import pandas as pd

raster_info = []

for _, row in inventory[inventory["Extension"].isin([".tif", ".tiff"])].iterrows():

    file_path = ROOT / row["Relative path"]

    try:
        with rasterio.open(file_path) as src:

            raster_info.append({
                "Band": row["Band"],
                "Filename": row["Filename"],
                "Path": row["Relative path"],
                "Width": src.width,
                "Height": src.height,
                "Bands": src.count,
                "Data type": str(src.dtypes),
                "CRS": str(src.crs),
                "Resolution": str(src.res),
                "NoData": src.nodata,
                "Bounds": str(src.bounds)
            })

    except Exception as e:
        print(f"Could not inspect {file_path}: {e}")

raster_audit = pd.DataFrame(raster_info)

pd.set_option("display.max_colwidth", 100)
display(raster_audit)

,Band,Filename,Path,Width,Height,Bands,Data type,CRS,Resolution,NoData,Bounds
0,L_BAND,24_HHHH_gamma0_db.tif,L_BAND/L BAND/extracted features/24_HHHH_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
1,L_BAND,24_HVHV_gamma0_db.tif,L_BAND/L BAND/extracted features/24_HVHV_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
2,L_BAND,18_HHHH_gamma0_db.tif,L_BAND/L BAND/extracted features/18_HHHH_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
3,L_BAND,18_HVHV_gamma0_db.tif,L_BAND/L BAND/extracted features/18_HVHV_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
4,L_BAND,HHHH_gamma0_db.tif,L_BAND/L BAND/extracted features/HHHH_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
5,L_BAND,HVHV_gamma0_db.tif,L_BAND/L BAND/extracted features/HVHV_gamma0_db.tif,2126,2006,1,"('float32',)",EPSG:32645,"(10.0, 10.0)",-9999.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
6,L_BAND,water_evidence_score.tif,L_BAND/L BAND/classification/20260819/water_evidence_score.tif,2126,2006,1,"('uint8',)",EPSG:32645,"(10.0, 10.0)",255.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
7,L_BAND,dark_candidate_consensus.tif,L_BAND/L BAND/classification/20260819/dark_candidate_consensus.tif,2126,2006,1,"('uint8',)",EPSG:32645,"(10.0, 10.0)",255.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
8,L_BAND,water_mask_binary.tif,L_BAND/L BAND/classification/20260819/water_mask_binary.tif,2126,2006,1,"('uint8',)",EPSG:32645,"(10.0, 10.0)",255.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"
9,L_BAND,water_mask_3class.tif,L_BAND/L BAND/classification/20260819/water_mask_3class.tif,2126,2006,1,"('uint8',)",EPSG:32645,"(10.0, 10.0)",255.0,"BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)"


In [7]:
import numpy as np
import rasterio

files_to_inspect = [
    "L_BAND/L BAND/extracted features/HHHH_gamma0_db.tif",
    "L_BAND/L BAND/extracted features/HVHV_gamma0_db.tif",
    "S_BAND/classification/20260819/water_mask.tif",
    "S_BAND/validation/aligned_classification/20260819/aligned_water_mask.tif",
    "S_BAND/shared/reference_labels_20260819.tif"
]

for relative_path in files_to_inspect:

    path = ROOT / relative_path

    with rasterio.open(path) as src:

        data = src.read(1, masked=True)
        values = data.compressed()

        print("\n" + "=" * 65)
        print("FILE:", relative_path)
        print("Shape:", data.shape)
        print("Valid pixels:", len(values))

        if len(values) > 0:
            print("Minimum:", values.min())
            print("Maximum:", values.max())
            print("Mean:", values.mean())
            print("Unique values (first 20):",
                  np.unique(values)[:20])


FILE: L_BAND/L BAND/extracted features/HHHH_gamma0_db.tif
Shape: (2006, 2126)
Valid pixels: 4264756
Minimum: -31.106596
Maximum: 33.13096
Mean: -11.003852
Unique values (first 20): [-31.106596 -30.065325 -30.041737 -29.81595  -29.799667 -29.779568
 -29.77219  -29.638199 -29.602385 -29.349861 -29.259869 -29.222809
 -28.978771 -28.90754  -28.861141 -28.85487  -28.828108 -28.763178
 -28.718864 -28.6981  ]

FILE: L_BAND/L BAND/extracted features/HVHV_gamma0_db.tif
Shape: (2006, 2126)
Valid pixels: 4264756
Minimum: -36.076866
Maximum: 25.348637
Mean: -18.842411
Unique values (first 20): [-36.076866 -35.698277 -35.54712  -35.40402  -34.873234 -34.85296
 -34.765415 -34.76086  -34.72388  -34.67851  -34.64552  -34.50226
 -34.474335 -34.456604 -34.367287 -34.354523 -34.329285 -34.249146
 -34.22237  -34.210537]

FILE: S_BAND/classification/20260819/water_mask.tif
Shape: (33120, 34200)
Valid pixels: 626238401
Minimum: 0
Maximum: 1
Mean: 0.13373121460815687
Unique values (first 20): [0 1]

FILE: S

In [8]:
import numpy as np
import rasterio

files = {
    "S original mask": "S_BAND/classification/20260819/water_mask.tif",
    "S aligned mask": "S_BAND/validation/aligned_classification/20260819/aligned_water_mask.tif",
    "Reference labels": "S_BAND/shared/reference_labels_20260819.tif"
}

for name, relative_path in files.items():

    with rasterio.open(ROOT / relative_path) as src:
        data = src.read(1, masked=True)
        values, counts = np.unique(data.compressed(), return_counts=True)

        print(f"\n{name}")
        total = counts.sum()

        for value, count in zip(values, counts):
            print(f"Value {value}: {count:,} pixels ({count/total*100:.2f}%)")

        print("Total valid pixels:", f"{total:,}")
        print("NoData pixels:", f"{np.ma.getmaskarray(data).sum():,}")


S original mask
Value 0: 542,490,779 pixels (86.63%)
Value 1: 83,747,622 pixels (13.37%)
Total valid pixels: 626,238,401
NoData pixels: 506,465,599

S aligned mask
Value 0: 3,839,342 pixels (89.94%)
Value 1: 429,547 pixels (10.06%)
Total valid pixels: 4,268,889
NoData pixels: 0

Reference labels
Value 0: 670,892 pixels (15.72%)
Value 1: 185,239 pixels (4.34%)
Value 255: 3,412,758 pixels (79.94%)
Total valid pixels: 4,268,889
NoData pixels: 0


In [9]:
import rasterio

files = {
    "L_HH": "L_BAND/L BAND/extracted features/HHHH_gamma0_db.tif",
    "L_HV": "L_BAND/L BAND/extracted features/HVHV_gamma0_db.tif",
    "S_mask": "S_BAND/validation/aligned_classification/20260819/aligned_water_mask.tif",
    "Reference": "S_BAND/shared/reference_labels_20260819.tif"
}

datasets = {}

for name, relative_path in files.items():

    with rasterio.open(ROOT / relative_path) as src:

        datasets[name] = {
            "width": src.width,
            "height": src.height,
            "crs": src.crs,
            "transform": src.transform,
            "bounds": src.bounds,
            "resolution": src.res
        }

        print("\n", name)
        print("Dimensions:", src.width, "x", src.height)
        print("CRS:", src.crs)
        print("Transform:", src.transform)
        print("Bounds:", src.bounds)

print("\n" + "=" * 60)
print("ALIGNMENT CHECKS")

for a, b in [("L_HH", "L_HV"), ("S_mask", "Reference")]:
    A, B = datasets[a], datasets[b]

    print(f"\n{a} vs {b}")
    print("Same dimensions:", (A["width"], A["height"]) == (B["width"], B["height"]))
    print("Same CRS:", A["crs"] == B["crs"])
    print("Same transform:", A["transform"] == B["transform"])
    print("Same bounds:", A["bounds"] == B["bounds"])


 L_HH
Dimensions: 2126 x 2006
CRS: EPSG:32645
Transform: | 10.00, 0.00, 439110.00|
| 0.00,-10.00, 2830440.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)

 L_HV
Dimensions: 2126 x 2006
CRS: EPSG:32645
Transform: | 10.00, 0.00, 439110.00|
| 0.00,-10.00, 2830440.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=439110.0, bottom=2810380.0, right=460370.0, top=2830440.0)

 S_mask
Dimensions: 2127 x 2007
CRS: EPSG:32645
Transform: | 10.00, 0.00, 439110.00|
| 0.00,-10.00, 2830450.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=439110.0, bottom=2810380.0, right=460380.0, top=2830450.0)

 Reference
Dimensions: 2127 x 2007
CRS: EPSG:32645
Transform: | 10.00, 0.00, 439110.00|
| 0.00,-10.00, 2830450.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=439110.0, bottom=2810380.0, right=460380.0, top=2830450.0)

ALIGNMENT CHECKS

L_HH vs L_HV
Same dimensions: True
Same CRS: True
Same transform: True
Same bounds: True

S_mask vs Reference
Same

In [10]:
import json

files_to_read = [
    "S_BAND/shared/common_grid_specification.json",
    "S_BAND/metadata/acquisition_details.json",
    "L_BAND/L BAND/metadata/run_summary.json"
]

for relative_path in files_to_read:

    path = ROOT / relative_path

    print("\n" + "=" * 80)
    print("FILE:", relative_path)

    with open(path, "r") as f:
        content = json.load(f)

    print(json.dumps(content, indent=4))


FILE: S_BAND/shared/common_grid_specification.json
{
    "description": "Common Sentinel-2 reference raster grid used for spatial comparison with NISAR S-band.",
    "reference_grid": {
        "crs": "EPSG:32645",
        "width": 2127,
        "height": 2007,
        "resolution": [
            10.0,
            10.0
        ],
        "transform": [
            10.0,
            0.0,
            439110.0,
            0.0,
            -10.0,
            2830450.0,
            0.0,
            0.0,
            1.0
        ],
        "bounds": [
            439110.0,
            2810380.0,
            460380.0,
            2830450.0
        ]
    },
    "reference_dates": [
        "20260819",
        "20260924",
        "20261018"
    ],
    "pixel_labels": {
        "0": "Non-water",
        "1": "Water",
        "255": "Invalid; excluded from validation"
    },
    "alignment": {
        "reference_grid_source": "Sentinel-2 GeoTIFF",
        "comparison": "Spatial comparison must u

In [11]:
# Find all L-band feature files
feature_folder = ROOT / "L_BAND" / "L BAND" / "extracted features"

print("L-BAND FEATURE FILES")
print("=" * 70)

for path in sorted(feature_folder.glob("*")):
    if path.is_file():
        print(f"{path.name:35} {path.stat().st_size / (1024**2):.2f} MB")

L-BAND FEATURE FILES
18_HHHH_gamma0_db.tif               14.31 MB
18_HVHV_gamma0_db.tif               13.81 MB
24_HHHH_gamma0_db.tif               14.34 MB
24_HVHV_gamma0_db.tif               13.85 MB
HHHH_gamma0_db.tif                  14.30 MB
HVHV_gamma0_db.tif                  13.79 MB


In [12]:
import json

dates = ["20260819", "20260924", "20261018"]

for date in dates:

    report_path = (
        ROOT / "L_BAND" / "L BAND" /
        "classification" / date /
        "thresholds_and_scene_report.json"
    )

    print("\n" + "=" * 75)
    print("ACQUISITION:", date)

    if report_path.exists():
        with open(report_path, "r") as f:
            report = json.load(f)

        print(json.dumps(report, indent=4))
    else:
        print("Report not found:", report_path)


ACQUISITION: 20260819
{
    "date": "20260819",
    "band": "NISAR L-band",
    "input_backscatter": "linear Gamma0 (gamma-zero) power",
    "dB_conversion": "10*log10(Gamma0)",
    "gamma0_to_sigma0_conversion": "NOT PERFORMED",
    "method": "per-date Otsu HH/HV + dual-channel consensus + 5x5 texture + 3x3 support",
    "classification_thresholds": {
        "date": "20260819",
        "HH_Otsu_threshold_dB": -12.099112274330107,
        "HV_Otsu_threshold_dB": -18.981970400321295,
        "HH_texture_threshold_dB_std": 2.5016016960144043,
        "HV_texture_threshold_dB_std": 2.258007049560547,
        "HH_Otsu_separability_diagnostic": 0.6913456780831339,
        "HV_Otsu_separability_diagnostic": 0.7037753135793349,
        "HH_Otsu_sample_size": 500000,
        "HV_Otsu_sample_size": 500000,
        "HH_histogram_range_low_dB": -22.920991516113283,
        "HH_histogram_range_high_dB": 1.2746511554718079,
        "HV_histogram_range_low_dB": -28.951543674468994,
        "HV_his

In [13]:
h5_inventory_path = (
    ROOT / "L_BAND" / "L BAND" /
    "metadata" / "hdf5_dataset_inventory.csv"
)

h5_inventory = pd.read_csv(h5_inventory_path)

print("Columns:", h5_inventory.columns.tolist())
display(h5_inventory.head(20))

Columns: ['file', 'dataset_path', 'shape', 'dtype', 'chunks', 'compression', 'attributes']


,file,dataset_path,shape,dtype,chunks,compression,attributes
0,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/HHHH,"(33696, 34344)",float32,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""_FillValue"": NaN..."
1,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/HVHV,"(33696, 34344)",float32,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""_FillValue"": NaN..."
2,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/inputDataExceptionMask,"(33696, 34344)",uint8,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""description"": ""B..."
3,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/listOfCovarianceTerms,"(2,)",|S4,NaN,NaN,"{""description"": ""List of processed covariance terms""}"
4,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/listOfPolarizations,"(2,)",|S2,NaN,NaN,"{""description"": ""List of processed polarization layers with frequency A""}"
5,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/mask,"(33696, 34344)",uint8,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""_FillValue"": 255..."
6,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/numberOfLooks,"(33696, 34344)",float32,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""_FillValue"": NaN..."
7,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/numberOfSubSwaths,(),uint8,NaN,NaN,"{""description"": ""Number of swaths of continuous imagery, due to transmit gaps"", ""units"": ""1""}"
8,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/projection,(),uint32,NaN,NaN,"{""description"": ""Product map grid projection: EPSG code, with additional projection information ..."
9,NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5,/science/LSAR/GCOV/grids/frequencyA/rtcGammaToSigmaFactor,"(33696, 34344)",float32,"(512, 512)",gzip,"{""DIMENSION_LIST"": [""[<HDF5 object reference>]"", ""[<HDF5 object reference>]""], ""_FillValue"": NaN..."


In [14]:
h5_files = list(ROOT.rglob("*.h5")) + list(ROOT.rglob("*.hdf5"))

print("HDF5 files found:", len(h5_files))

for path in h5_files:
    print("\nFile:", path.name)
    print("Path:", path)
    print("Size (GB):", round(path.stat().st_size / (1024**3), 2))

HDF5 files found: 4

File: compound-dtype-complex.h5
Path: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/lib/python3.12/site-packages/h5py/tests/data_files/compound-dtype-complex.h5
Size (GB): 0.0

File: vlen_string_s390x.h5
Path: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/lib/python3.12/site-packages/h5py/tests/data_files/vlen_string_s390x.h5
Size (GB): 0.0

File: vlen_string_dset_utc.h5
Path: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/lib/python3.12/site-packages/h5py/tests/data_files/vlen_string_dset_utc.h5
Size (GB): 0.0

File: vlen_string_dset.h5
Path: /Users/shrenishetty/Documents/NISAR_L+S/NISAR/.venv/lib/python3.12/site-packages/h5py/tests/data_files/vlen_string_dset.h5
Size (GB): 0.0


In [15]:
frequency_inventory = h5_inventory[
    h5_inventory["dataset_path"].str.contains(
        "frequencyA|frequencyB",
        case=False,
        regex=True
    )
]

display(
    frequency_inventory[
        ["file", "dataset_path", "shape", "dtype", "attributes"]
    ].to_string(index=False)
)

'                                                                                          file                                                                                               dataset_path          shape     dtype                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [16]:
frequency_inventory = h5_inventory[
    h5_inventory["dataset_path"].str.contains(
        "frequencyA|frequencyB",
        case=False,
        regex=True
    )
]

display(
    frequency_inventory[
        ["file", "dataset_path", "shape", "dtype", "attributes"]
    ].to_string(index=False)
)

'                                                                                          file                                                                                               dataset_path          shape     dtype                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [17]:
from pathlib import Path

h5_files = []

for path in ROOT.rglob("*"):
    if ".venv" in path.parts:
        continue

    if path.is_file() and path.suffix.lower() in [".h5", ".hdf5"]:
        h5_files.append(path)

print("NISAR HDF5 files found:", len(h5_files))

for path in h5_files:
    print(path)

NISAR HDF5 files found: 0


In [18]:
import h5py

if not h5_files:
    print("No HDF5 files found in repository.")
else:

    h5_path = h5_files[0]

    with h5py.File(h5_path, "r") as f:

        print("HDF5:", h5_path.name)

        for group_name in ["frequencyA", "frequencyB"]:

            group = f[
                f"/science/LSAR/GCOV/grids/{group_name}"
            ]

            print("\n", "=" * 60)
            print("GROUP:", group_name)

            for key in [
                "HHHH",
                "HVHV",
                "listOfCovarianceTerms",
                "listOfPolarizations",
                "xCoordinateSpacing",
                "yCoordinateSpacing",
                "projection",
                "mask",
                "inputDataExceptionMask"
            ]:

                if key in group:
                    obj = group[key]

                    if isinstance(obj, h5py.Dataset):
                        print("\n", key)
                        print("Shape:", obj.shape)
                        print("Attributes:")

                        for attr, value in obj.attrs.items():
                            print(f"  {attr}: {value}")

                        if obj.shape == ():
                            print("Value:", obj[()])
                        elif key in [
                            "listOfCovarianceTerms",
                            "listOfPolarizations",
                            "xCoordinateSpacing",
                            "yCoordinateSpacing",
                            "projection"
                        ]:
                            print("Value:", obj[()])

No HDF5 files found in repository.


In [19]:
import pandas as pd
import json

inventory_path = (
    ROOT / "L_BAND" / "L BAND" /
    "metadata" / "hdf5_dataset_inventory.csv"
)

h5_inventory = pd.read_csv(inventory_path)

# Keep only the two frequency groups
frequency_inventory = h5_inventory[
    h5_inventory["dataset_path"].str.contains(
        "frequencyA|frequencyB",
        case=False,
        regex=True
    )
].copy()

# Display the important datasets
important_datasets = [
    "HHHH",
    "HVHV",
    "mask",
    "inputDataExceptionMask",
    "numberOfLooks",
    "rtcGammaToSigmaFactor",
    "projection",
    "xCoordinateSpacing",
    "yCoordinateSpacing",
    "xCoordinates",
    "yCoordinates",
    "listOfCovarianceTerms",
    "listOfPolarizations"
]

frequency_inventory["dataset_name"] = (
    frequency_inventory["dataset_path"].str.split("/").str[-1]
)

display(
    frequency_inventory[
        frequency_inventory["dataset_name"].isin(important_datasets)
    ][
        ["dataset_path", "shape", "dtype", "attributes"]
    ].to_string(index=False)
)

'                                                                                         dataset_path          shape   dtype                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [20]:
h5_inventory["file"].dropna().unique()

<StringArray>
['NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5',
 'NISAR_L2_PR_GCOV_031_098_A_014_4005_DHDH_A_20260924T233810_20260924T233845_P05023_N_F_J_001.h5',
 'NISAR_L2_PR_GCOV_033_098_A_014_4005_DHDH_A_20261018T233811_20261018T233845_P05023_N_F_J_001.h5']
Length: 3, dtype: str

In [21]:
print("Unique HDF5 products:",
      h5_inventory["file"].nunique())

print("\nProducts:")
for name in h5_inventory["file"].dropna().unique():
    print(name)

Unique HDF5 products: 3

Products:
NISAR_L2_PR_GCOV_028_098_A_014_4005_DHDH_A_20260819T233811_20260819T233845_P05023_N_F_J_001.h5
NISAR_L2_PR_GCOV_031_098_A_014_4005_DHDH_A_20260924T233810_20260924T233845_P05023_N_F_J_001.h5
NISAR_L2_PR_GCOV_033_098_A_014_4005_DHDH_A_20261018T233811_20261018T233845_P05023_N_F_J_001.h5


In [22]:
# Extract the acquisition date from each filename
h5_inventory["date"] = (
    h5_inventory["file"].str.extract(r"(\d{8})T")[0]
)

# Extract frequency group
h5_inventory["frequency"] = (
    h5_inventory["dataset_path"]
    .str.extract(r"/(frequencyA|frequencyB)/")[0]
)

# Extract dataset name
h5_inventory["dataset"] = (
    h5_inventory["dataset_path"].str.split("/").str[-1]
)

# Summarise important datasets
summary = (
    h5_inventory[
        h5_inventory["dataset"].isin([
            "HHHH",
            "HVHV",
            "mask",
            "numberOfLooks",
            "rtcGammaToSigmaFactor"
        ])
    ]
    .groupby(["date", "frequency", "dataset"])
    .agg(
        shape=("shape", "first"),
        dtype=("dtype", "first")
    )
    .reset_index()
)

display(summary.to_string(index=False))

'    date  frequency               dataset          shape   dtype\n20260819 frequencyA                  HHHH (33696, 34344) float32\n20260819 frequencyA                  HVHV (33696, 34344) float32\n20260819 frequencyA                  mask (33696, 34344)   uint8\n20260819 frequencyA         numberOfLooks (33696, 34344) float32\n20260819 frequencyA rtcGammaToSigmaFactor (33696, 34344) float32\n20260819 frequencyB                  HHHH   (4212, 4293) float32\n20260819 frequencyB                  HVHV   (4212, 4293) float32\n20260819 frequencyB                  mask   (4212, 4293)   uint8\n20260819 frequencyB         numberOfLooks   (4212, 4293) float32\n20260819 frequencyB rtcGammaToSigmaFactor   (4212, 4293) float32\n20260924 frequencyA                  HHHH (33696, 34344) float32\n20260924 frequencyA                  HVHV (33696, 34344) float32\n20260924 frequencyA                  mask (33696, 34344)   uint8\n20260924 frequencyA         numberOfLooks (33696, 34344) float32\n20260924 

In [23]:
# Search the complete inventory for frequency-related metadata

keywords = [
    "frequency",
    "radarFrequency",
    "centerFrequency",
    "center_frequency",
    "wavelength",
    "frequencyA",
    "frequencyB"
]

for keyword in keywords:
    matches = h5_inventory[
        h5_inventory["dataset_path"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    print(f"\n--- {keyword} ---")
    print(matches[["file", "dataset_path", "shape", "attributes"]]
          .to_string(index=False))


--- frequency ---
                                                                                          file                                                                                               dataset_path          shape                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [24]:
from pathlib import Path

h5_files = list(ROOT.rglob("*.h5")) + list(ROOT.rglob("*.hdf5"))

print("TOTAL HDF5 FILES:", len(h5_files))

print("\nS-BAND FILES:")
s_h5_files = [
    p for p in h5_files
    if p.name.startswith("NISAR_S2")
]

for p in sorted(s_h5_files):
    print("\nFilename:", p.name)
    print("Location:", p.relative_to(ROOT))
    print("Size (GB):", round(p.stat().st_size / (1024**3), 2))

print("\nNUMBER OF S-BAND FILES:", len(s_h5_files))

TOTAL HDF5 FILES: 4

S-BAND FILES:

NUMBER OF S-BAND FILES: 0


In [25]:
git status

SyntaxError: invalid syntax (3528599804.py, line 1)